# Lab: The Quantum Fourier Transform with Qiskit

In this lab we implement the **quantum Fourier transform (QFT)**, the quantum analogue of the discrete Fourier transform and a key subroutine in many quantum algorithms, including Shor's algorithm and quantum phase estimation.

For an $n$-qubit register with $N = 2^n$, the QFT acts on computational basis states as

$$\text{QFT}|x\rangle = \frac{1}{\sqrt{N}} \sum_{y=0}^{N-1} e^{2\pi i x y / N} |y\rangle.$$

This is exactly the discrete Fourier transform applied to the $2^n$ amplitudes, but implemented with a circuit of only $O(n^2)$ gates. In this lab we will:

1. Warm up by estimating a single-qubit phase with one-, two-, and three-stage Ramsey circuits.
2. Build the QFT circuit from Hadamard and controlled-phase gates.
3. Check it against the classical discrete Fourier transform for every basis state.
4. Build the inverse QFT and verify that it undoes the QFT.
5. Use the inverse QFT inside quantum phase estimation.

> **Convention:** Qiskit uses **little-endian** ordering. Qubit 0 is the least significant bit, and the statevector index of $|x\rangle$ is $\sum_{k=0}^{n-1} x_k 2^k$. All tests below follow this convention.


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator

print("Qiskit imported successfully!")


## Warm-Up: Estimating a Single-Qubit Phase

Before building the QFT, let's estimate the phase of a single-qubit rotation. Let

$$U = P(\theta) = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\theta} \end{pmatrix}$$

be an **unknown** phase rotation, with a promise that will restrict $\theta$ to a small set of values. The basic tool is the **Ramsey sequence**:

$$|0\rangle \xrightarrow{H} |+\rangle \xrightarrow{U} \frac{|0\rangle + e^{i\theta}|1\rangle}{\sqrt{2}} \xrightarrow{H} \text{measure}.$$

In Qiskit this is built with a Hadamard, the unknown unitary, and a second Hadamard:

```python
circuit = QuantumCircuit(1)
circuit.h(0)
circuit.compose(unitary, inplace=True)
circuit.h(0)
```

The probability of measuring `0` is $\cos^2(\theta/2)$. In particular, $\theta = 0$ gives `0` with certainty and $\theta = \pi$ gives `1` with certainty.


### Warm-Up Helpers

The warm-up exercises use two lists of promised phases:

- `PHASES = [0, pi/2, pi, 3*pi/2]` for the two-stage estimator.
- `EIGHT_PHASES = [0, pi/4, ..., 7*pi/4]` for the three-stage estimator.

Every promised measurement in these warm-ups is deterministic, so you can read a result directly from a circuit's statevector:

```python
outcome = int(np.argmax(Statevector(circuit).probabilities()))
```

The guardrail `validate_one_qubit_unitary(unitary)` checks that an input circuit acts on exactly one qubit.


In [ ]:
PHASES = [0.0, np.pi / 2, np.pi, 3 * np.pi / 2]
EIGHT_PHASES = [k * np.pi / 4 for k in range(8)]


def validate_one_qubit_unitary(unitary: QuantumCircuit) -> None:
    """Guardrail: checks that unitary is a one-qubit circuit."""
    assert isinstance(unitary, QuantumCircuit), "unitary must be a QuantumCircuit."
    assert unitary.num_qubits == 1, "unitary must act on exactly one qubit."


### Warm-Up 1: Distinguishing $\theta = 0$ from $\theta = \pi$

Implement `measure_phase_0_or_pi(unitary: QuantumCircuit) -> int`.

- `unitary`: a one-qubit circuit implementing $U = P(\theta)$, promised to have $\theta \in \{0, \pi\}$.
- Returns `0` for $\theta = 0$ and `1` for $\theta = \pi$.


In [ ]:
def measure_phase_0_or_pi(unitary: QuantumCircuit) -> int:
    """
    Measures whether the phase of unitary is 0 or pi.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta), with
            theta in {0, pi}.

    Returns:
        int: 0 if theta = 0, or 1 if theta = pi.
    """
    validate_one_qubit_unitary(unitary)

    # TODO: build the Ramsey sequence and return the measurement outcome
    pass


The cell below checks both promised phases over several repeated runs.


In [ ]:
for trial in range(5):
    for theta, expected in [(0.0, 0), (np.pi, 1)]:
        unitary = QuantumCircuit(1)
        unitary.p(theta, 0)

        observed = measure_phase_0_or_pi(unitary)
        assert observed == expected, (
            f"trial={trial}, theta={theta}: function returned {observed}, "
            f"expected {expected}."
        )

print("Warm-up 1 passed for repeated runs!")


### Warm-Up 2: Distinguishing $\{0, \pi\}$ from $\{\pi/2, 3\pi/2\}$

Now suppose $\theta \in \{0, \pi/2, \pi, 3\pi/2\}$ but we only need to know which **pair** contains it.

Implement `measure_phase_quadrant(unitary: QuantumCircuit) -> int`.

- `unitary`: a one-qubit circuit implementing $U = P(\theta)$ with $\theta$ in the four-phase set.
- Returns `0` if $\theta \in \{0, \pi\}$ and `1` if $\theta \in \{\pi/2, 3\pi/2\}$.


In [ ]:
def measure_phase_quadrant(unitary: QuantumCircuit) -> int:
    """
    Measures whether theta is in {0, pi} or in {pi/2, 3*pi/2}.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta), with
            theta in {0, pi/2, pi, 3*pi/2}.

    Returns:
        int: 0 if theta is in {0, pi}, or 1 if theta is in {pi/2, 3*pi/2}.
    """
    validate_one_qubit_unitary(unitary)

    # TODO: apply U twice inside a Ramsey sequence and return the outcome
    pass


The cell below checks all four promised phases over several seeds.


In [ ]:
for seed in range(5):
    np.random.seed(seed)

    for k, theta in enumerate(PHASES):
        expected = 0 if k % 2 == 0 else 1

        unitary = QuantumCircuit(1)
        unitary.p(theta, 0)

        observed = measure_phase_quadrant(unitary)
        assert observed == expected, (
            f"seed={seed}, theta={theta}: function returned {observed}, "
            f"expected {expected}."
        )

print("Warm-up 2 passed for multiple seeds!")


### Warm-Up 3: A Two-Stage, Single-Shot Estimator

We now distinguish all four phases with two Ramsey sequences, measuring each one once. The first sequence is `measure_phase_quadrant` from Warm-Up 2. Its result tells us which pair contains $\theta$:

- first result `0`: $\theta \in \{0, \pi\}$.
- first result `1`: $\theta \in \{\pi/2, 3\pi/2\}$.

For the second sequence we reuse Warm-Up 1, but when the first result is `1` we shift the phase by $-\pi/2$ so that the remaining phase is again in $\{0, \pi\}$. Implement the two-stage estimator twice:

- `measure_phase_single_shot_modify(unitary)`: implement the second sequence by building a **phase-shifted copy of $U$** (compose a `p(-np.pi/2, 0)` gate into a copy of `unitary`).
- `measure_phase_single_shot_controlled(unitary)`: implement the second sequence with the original $U$, then append a `p(-np.pi/2, 0)` gate only when the first result is `1`.

Both return the two-bit string `b0 b1`, where `b0` is the first-sequence result and `b1` is the second-sequence result. The promised phases map to:

| $\theta$ | result |
|:---:|:---:|
| $0$ | `00` |
| $\pi/2$ | `10` |
| $\pi$ | `01` |
| $3\pi/2$ | `11` |

> **Hint:** in both versions, build the second Ramsey sequence manually: `H`, then the (possibly shifted) $U$, then `H`. For the controlled version, the correction gate goes between $U$ and the final `H`.


#### Implementation A: Modify the Phase Rotation

Build the second sequence from a phase-shifted copy of $U$.


In [ ]:
def measure_phase_single_shot_modify(unitary: QuantumCircuit) -> str:
    """
    Runs the two-stage estimator, implementing the second stage by
    modifying the phase rotation performed by U.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta), with
            theta in {0, pi/2, pi, 3*pi/2}.

    Returns:
        str: A two-bit string b0 b1, where b0 is the quadrant and b1
            distinguishes the two phases within that quadrant.
    """
    validate_one_qubit_unitary(unitary)

    # TODO: run the first stage, then the second stage using a
    # phase-shifted copy of unitary
    pass


#### Implementation B: Classically Controlled Correction

Use the original $U$ in the second sequence and append the correction gate only when the first result is `1`.


In [ ]:
def measure_phase_single_shot_controlled(unitary: QuantumCircuit) -> str:
    """
    Runs the two-stage estimator, implementing the second stage by
    applying the original U followed by a classically controlled phase gate.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta), with
            theta in {0, pi/2, pi, 3*pi/2}.

    Returns:
        str: A two-bit string b0 b1, where b0 is the quadrant and b1
            distinguishes the two phases within that quadrant.
    """
    validate_one_qubit_unitary(unitary)

    # TODO: run the first stage, then the second stage with a
    # classically controlled correction after unitary
    pass


The cell below checks both implementations against all four promised phases over several seeds. The two-bit results must match the table above.


In [ ]:
EXPECTED_RESULTS = ["00", "10", "01", "11"]

for seed in range(5):
    np.random.seed(seed)

    for k, theta in enumerate(PHASES):
        unitary = QuantumCircuit(1)
        unitary.p(theta, 0)

        expected = EXPECTED_RESULTS[k]

        modified = measure_phase_single_shot_modify(unitary)
        assert modified == expected, (
            f"seed={seed}, modify version for theta={theta} returned "
            f"{modified}, expected {expected}."
        )

        controlled = measure_phase_single_shot_controlled(unitary)
        assert controlled == expected, (
            f"seed={seed}, controlled version for theta={theta} returned "
            f"{controlled}, expected {expected}."
        )

print("Warm-up 3 passed for multiple seeds!")


## Warm-Up 4: Three-Stage Ramsey Interferometry

Now promise that

$$\theta = \frac{k\pi}{4}, \qquad k = 4b_2 + 2b_1 + b_0, \qquad b_i \in \{0, 1\},$$

so $\theta \in \{0, \pi/4, \ldots, 7\pi/4\}$. We will determine all three bits with three Ramsey stages. Each stage must test a phase guaranteed to be $0$ or $\pi$.

Implement the three stages separately below. Then combine them in two full implementations: one that **modifies the unitary directly**, and one that **keeps the original unitary and applies conditional phase rotations**.


### Warm-Up 4.1: Stage 1 — Measure $b_0$

Implement `measure_b0(unitary: QuantumCircuit) -> int`.

- Applying $U$ four times tests $4\theta = b_0\pi \pmod{2\pi}$.
- Return $b_0$, the least significant bit.


In [ ]:
def measure_b0(unitary: QuantumCircuit) -> int:
    """
    Measures b0, the least significant bit of theta = k * pi / 4.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta), with
            theta = k * pi / 4 for k in {0, ..., 7}.

    Returns:
        int: b0.
    """
    validate_one_qubit_unitary(unitary)

    # TODO: build the first Ramsey stage and return b0
    pass


The cell below checks the first stage for all eight promised phases.


In [ ]:
for k, theta in enumerate(EIGHT_PHASES):
    unitary = QuantumCircuit(1)
    unitary.p(theta, 0)

    expected = k % 2
    observed = measure_b0(unitary)
    assert observed == expected, (
        f"Stage 1 for theta={theta} returned {observed}, expected {expected}."
    )

print("Warm-up 4, stage 1 passed!")


### Warm-Up 4.2: Stage 2 — Measure $b_1$

Suppose stage 1 returned $b_0$. The second stage should test

$$2\left(\theta - \frac{b_0\pi}{4}\right) = 2\theta - \frac{b_0\pi}{2},$$

which is $0$ or $\pi$ modulo $2\pi$.

Implement this stage in two ways:

- `measure_b1_modified(unitary, b0)`: build a phase-shifted copy of $U$ so that applying it twice tests $b_1$.
- `measure_b1_controlled(unitary, b0)`: apply the original $U$ twice, then apply a phase rotation that removes the unwanted part of the phase.


In [ ]:
def measure_b1_modified(unitary: QuantumCircuit, b0: int) -> int:
    """
    Measures b1 by modifying the phase of the unitary directly.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta).
        b0: The result of stage 1, either 0 or 1.

    Returns:
        int: b1.
    """
    validate_one_qubit_unitary(unitary)
    assert b0 in (0, 1), "b0 must be 0 or 1."

    # TODO: build a phase-shifted copy of unitary and use it twice
    pass


In [ ]:
def measure_b1_controlled(unitary: QuantumCircuit, b0: int) -> int:
    """
    Measures b1 using the original unitary and a conditional phase rotation.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta).
        b0: The result of stage 1, either 0 or 1.

    Returns:
        int: b1.
    """
    validate_one_qubit_unitary(unitary)
    assert b0 in (0, 1), "b0 must be 0 or 1."

    # TODO: apply unitary twice, then a conditional phase rotation
    pass


The cell below checks both stage-2 implementations for all eight promised phases.


In [ ]:
for k, theta in enumerate(EIGHT_PHASES):
    unitary = QuantumCircuit(1)
    unitary.p(theta, 0)

    b0 = k % 2
    expected = (k // 2) % 2

    modified = measure_b1_modified(unitary, b0)
    assert modified == expected, (
        f"Stage 2 modified for theta={theta} returned {modified}, expected {expected}."
    )

    controlled = measure_b1_controlled(unitary, b0)
    assert controlled == expected, (
        f"Stage 2 controlled for theta={theta} returned {controlled}, expected {expected}."
    )

print("Warm-up 4, stage 2 passed!")


### Warm-Up 4.3: Stage 3 — Measure $b_2$

Suppose stages 1 and 2 returned $b_0$ and $b_1$. The third stage should test

$$\theta - \frac{b_0\pi}{4} - \frac{b_1\pi}{2},$$

which is $0$ or $\pi$ modulo $2\pi$.

Implement this stage in two ways:

- `measure_b2_modified(unitary, b0, b1)`: build a phase-shifted copy of $U$ and apply it once.
- `measure_b2_controlled(unitary, b0, b1)`: apply the original $U$ once, then apply a phase rotation that removes the unwanted part of the phase.


In [ ]:
def measure_b2_modified(unitary: QuantumCircuit, b0: int, b1: int) -> int:
    """
    Measures b2 by modifying the phase of the unitary directly.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta).
        b0: The result of stage 1, either 0 or 1.
        b1: The result of stage 2, either 0 or 1.

    Returns:
        int: b2.
    """
    validate_one_qubit_unitary(unitary)
    assert b0 in (0, 1), "b0 must be 0 or 1."
    assert b1 in (0, 1), "b1 must be 0 or 1."

    # TODO: build a phase-shifted copy of unitary and use it once
    pass


In [ ]:
def measure_b2_controlled(unitary: QuantumCircuit, b0: int, b1: int) -> int:
    """
    Measures b2 using the original unitary and a conditional phase rotation.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta).
        b0: The result of stage 1, either 0 or 1.
        b1: The result of stage 2, either 0 or 1.

    Returns:
        int: b2.
    """
    validate_one_qubit_unitary(unitary)
    assert b0 in (0, 1), "b0 must be 0 or 1."
    assert b1 in (0, 1), "b1 must be 0 or 1."

    # TODO: apply unitary once, then a conditional phase rotation
    pass


The cell below checks both stage-3 implementations for all eight promised phases.


In [ ]:
for k, theta in enumerate(EIGHT_PHASES):
    unitary = QuantumCircuit(1)
    unitary.p(theta, 0)

    b0 = k % 2
    b1 = (k // 2) % 2
    expected = k // 4

    modified = measure_b2_modified(unitary, b0, b1)
    assert modified == expected, (
        f"Stage 3 modified for theta={theta} returned {modified}, expected {expected}."
    )

    controlled = measure_b2_controlled(unitary, b0, b1)
    assert controlled == expected, (
        f"Stage 3 controlled for theta={theta} returned {controlled}, expected {expected}."
    )

print("Warm-up 4, stage 3 passed!")


### Warm-Up 4.4: Combine the Stages

Implement the full three-stage estimator in two ways:

- `measure_phase_three_stage_modified(unitary)`: call the three stage functions that modify $U$ directly.
- `measure_phase_three_stage_controlled(unitary)`: call the three stage functions that use conditional rotations.

Both return the three-bit string `b2 b1 b0`, so that `int(result, 2) == k`.


In [ ]:
def measure_phase_three_stage_modified(unitary: QuantumCircuit) -> str:
    """
    Runs the three-stage estimator using phase-shifted copies of U.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta), with
            theta = k * pi / 4 for k in {0, ..., 7}.

    Returns:
        str: The three-bit string b2 b1 b0 for k = 4*b2 + 2*b1 + b0.
    """
    validate_one_qubit_unitary(unitary)

    # TODO: combine the three stages that modify unitary directly
    pass


In [ ]:
def measure_phase_three_stage_controlled(unitary: QuantumCircuit) -> str:
    """
    Runs the three-stage estimator using the original U and conditional rotations.

    Args:
        unitary: A one-qubit circuit implementing U = P(theta), with
            theta = k * pi / 4 for k in {0, ..., 7}.

    Returns:
        str: The three-bit string b2 b1 b0 for k = 4*b2 + 2*b1 + b0.
    """
    validate_one_qubit_unitary(unitary)

    # TODO: combine the three stages that use conditional rotations
    pass


The cell below checks both full implementations for all eight promised phases.


In [ ]:
for k, theta in enumerate(EIGHT_PHASES):
    unitary = QuantumCircuit(1)
    unitary.p(theta, 0)

    expected = format(k, "03b")

    modified = measure_phase_three_stage_modified(unitary)
    assert modified == expected, (
        f"Modified estimator for theta={theta} returned {modified}, expected {expected}."
    )

    controlled = measure_phase_three_stage_controlled(unitary)
    assert controlled == expected, (
        f"Controlled estimator for theta={theta} returned {controlled}, expected {expected}."
    )

print("Warm-up 4 passed!")


### Helper Functions & Guardrails

The test cells below rely on three helpers:

- `validate_qubit_count(n)` checks that `n` is a positive integer.
- `basis_state_circuit(n, x)` prepares the computational basis state $|x\rangle$.
- `classical_qft(x, n)` returns the exact amplitudes $e^{2\pi i x y / N} / \sqrt{N}$ of the QFT applied to $|x\rangle$.


In [ ]:
def validate_qubit_count(n: int) -> None:
    """Guardrail: checks that n is a positive integer."""
    assert isinstance(n, int) and n >= 1, "n must be a positive integer."


def basis_state_circuit(n: int, x: int) -> QuantumCircuit:
    """Prepares the n-qubit computational basis state |x>."""
    validate_qubit_count(n)
    assert isinstance(x, int) and 0 <= x < 2 ** n, (
        f"x must be an integer in [0, 2**n), got {x}."
    )

    circuit = QuantumCircuit(n)
    for qubit in range(n):
        if (x >> qubit) & 1:
            circuit.x(qubit)
    return circuit


def classical_qft(x: int, n: int) -> np.ndarray:
    """Returns the exact statevector amplitudes of QFT|x>."""
    validate_qubit_count(n)
    assert isinstance(x, int) and 0 <= x < 2 ** n, (
        f"x must be an integer in [0, 2**n), got {x}."
    )

    N = 2 ** n
    return np.array([
        np.exp(2j * np.pi * x * y / N) / np.sqrt(N)
        for y in range(N)
    ])


## Task 1: The QFT Circuit

Writing $x = \sum_{k=0}^{n-1} x_k 2^k$, the QFT factorizes into a product over qubits:

$$\text{QFT}|x\rangle = \frac{1}{\sqrt{N}} \bigotimes_{k=0}^{n-1} \left( |0\rangle + e^{2\pi i x 2^k / N} |1\rangle \right).$$

This product form gives a circuit built from Hadamard gates and controlled-phase gates. The controlled-phase gate

$$CP(\theta) = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & e^{i\theta} \end{pmatrix}$$

adds a phase $\theta$ exactly when both qubits are $|1\rangle$. In Qiskit it is `circuit.cp(theta, control, target)`.


### Task 1.1: Implement the QFT

Implement `qft(n: int) -> QuantumCircuit`.

- `n`: number of qubits.
- Returns an `n`-qubit circuit implementing the QFT (with no measurements).

> **Hint:** loop over qubits from `n - 1` down to `0`. On qubit `i`, apply a Hadamard, then for each `j < i` apply `cp(2 * np.pi / 2 ** (i - j + 1), j, i)`. Finish by swapping qubit `i` with qubit `n - 1 - i` for the first half of the register, which reverses the qubit order.


In [ ]:
def qft(n: int) -> QuantumCircuit:
    """
    Builds the n-qubit quantum Fourier transform circuit.

    Args:
        n: Number of qubits.

    Returns:
        QuantumCircuit: An n-qubit circuit implementing the QFT.
    """
    validate_qubit_count(n)

    # TODO: build the QFT circuit
    pass


### Task 1.2: Test the QFT

The cell below checks that `qft(n)` matches the classical discrete Fourier transform for every basis state on 1, 2, 3, and 4 qubits, and that the circuit is unitary.


In [ ]:
for n in [1, 2, 3, 4]:
    assert Operator(qft(n)).is_unitary(), f"QFT for n={n} is not unitary."

    for x in range(2 ** n):
        circuit = basis_state_circuit(n, x)
        circuit.compose(qft(n), inplace=True)
        observed = Statevector(circuit).data
        expected = classical_qft(x, n)
        assert np.allclose(observed, expected, atol=1e-8), (
            f"QFT for n={n} failed on basis state x={x}: "
            f"observed {np.round(observed, 4)}, expected {np.round(expected, 4)}"
        )

print("QFT passed all tests!")


## Task 2: The Inverse QFT

The inverse QFT is the adjoint of the QFT:

$$\text{QFT}^{\dagger}|y\rangle = \frac{1}{\sqrt{N}} \sum_{x=0}^{N-1} e^{-2\pi i x y / N} |x\rangle.$$

Because the QFT circuit is unitary, the inverse is obtained by reversing the order of the gates and negating every phase angle. It is the key step that turns the phases collected in phase estimation into a readable bitstring.


### Task 2.1: Implement the Inverse QFT

Implement `inverse_qft(n: int) -> QuantumCircuit`.

- `n`: number of qubits.
- Returns an `n`-qubit circuit implementing $\text{QFT}^{\dagger}$ (with no measurements).

> **Hint:** Qiskit circuits have an `.inverse()` method. Alternatively, rebuild the QFT circuit with `-2 * np.pi` phase angles.


In [ ]:
def inverse_qft(n: int) -> QuantumCircuit:
    """
    Builds the n-qubit inverse quantum Fourier transform circuit.

    Args:
        n: Number of qubits.

    Returns:
        QuantumCircuit: An n-qubit circuit implementing QFT^dagger.
    """
    validate_qubit_count(n)

    # TODO: build the inverse QFT circuit
    pass


### Task 2.2: Test the Inverse QFT

The cell below checks that `inverse_qft(n)` is the adjoint of `qft(n)` and that applying the QFT followed by the inverse QFT returns every basis state unchanged, for 1 through 5 qubits.


In [ ]:
for n in [1, 2, 3, 4, 5]:
    forward = Operator(qft(n))
    backward = Operator(inverse_qft(n))
    assert backward.equiv(forward.adjoint()), (
        f"inverse_qft(n={n}) is not the adjoint of qft(n={n})."
    )

    for x in range(2 ** n):
        circuit = basis_state_circuit(n, x)
        circuit.compose(qft(n), inplace=True)
        circuit.compose(inverse_qft(n), inplace=True)
        assert Statevector(circuit).equiv(Statevector(basis_state_circuit(n, x))), (
            f"QFT followed by inverse_qft did not restore basis state x={x} for n={n}."
        )

print("Inverse QFT passed all tests!")


## Task 3: Application — Quantum Phase Estimation

Quantum phase estimation estimates the phase $\varphi$ in the eigenvalue equation

$$U|\psi\rangle = e^{2\pi i \varphi}|\psi\rangle, \qquad 0 \le \varphi < 1.$$

Unlike the warm-up, $U$ can now be **any** unitary: it acts on a separate target register that is prepared in an eigenstate $|\psi\rangle$, while the phase is read out from the counting register with the inverse QFT.

We take $U = P(2\pi\varphi)$, the phase gate, with eigenstate $|\psi\rangle = |1\rangle$. For an $n$-bit phase $\varphi = x / 2^n$, phase estimation should return the bitstring for $x$.

The circuit uses $n$ counting qubits (indices $0, \ldots, n-1$) and one target qubit (index $n$):

1. Prepare the target in $|1\rangle$.
2. Apply a Hadamard to every counting qubit.
3. For each counting qubit $k$, apply controlled-$U^{2^k}$ to the target. Since $U$ is a phase gate, this is `cp(2 * np.pi * phi * 2 ** k, k, n)`.
4. Apply the inverse QFT to the counting qubits.
5. Read out the counting register.

The most probable measurement outcome is the binary representation of $\varphi$.


### Task 3.1: Implement Phase Estimation

Implement `phase_estimation(n: int, phi: float) -> str`.

- `n`: number of counting qubits.
- `phi`: the phase, which must equal `x / 2 ** n` for some integer `x`.
- Returns the `n`-bit string for `x`, so that `phi = int(result, 2) / 2 ** n`.

> **Hint:** build an `(n + 1)`-qubit circuit and compose `inverse_qft(n)` on the first `n` qubits with `circuit.compose(inverse_qft(n), qubits=list(range(n)), inplace=True)`. After building the circuit, use `Statevector(circuit).probabilities_dict()` to find the most probable bitstring. In Qiskit's display order the target qubit is the **first** character, so the counting register is `best[1:]`.


In [ ]:
def phase_estimation(n: int, phi: float) -> str:
    """
    Estimates the phase phi of U = P(2 * pi * phi) using n counting qubits.

    Args:
        n: Number of counting qubits.
        phi: The phase, with phi = x / 2**n for an integer x in [0, 2**n).

    Returns:
        str: The n-bit string for x, so that phi = int(result, 2) / 2 ** n.
    """
    validate_qubit_count(n)
    assert isinstance(phi, (int, float)) and 0 <= phi < 1, (
        "phi must be a number in [0, 1)."
    )
    assert np.isclose(phi * 2 ** n, round(phi * 2 ** n)), (
        f"phi={phi} is not exactly representable with n={n} bits."
    )

    # TODO: build and simulate the phase estimation circuit
    pass


### Task 3.2: Final Test

The cell below runs phase estimation for every exactly representable phase on 1, 2, 3, 4, and 5 counting qubits. It must pass for the lab to be complete.


In [ ]:
for n in [1, 2, 3, 4, 5]:
    for x in range(2 ** n):
        phi = x / 2 ** n
        expected = format(x, f"0{n}b")
        observed = phase_estimation(n, phi)
        assert observed == expected, (
            f"Phase estimation with n={n} qubits failed for phi={phi}: "
            f"got '{observed}', expected '{expected}'."
        )

print("Phase estimation passed all tests!")
